In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA

df = pd.read_csv("Chocolate_Sales.csv")

In [2]:
# Check data types
print("Data Types:\n", df.dtypes)

# Check for missing values
print("\nMissing Values:\n", df.isna().sum())

Data Types:
 Sales Person     object
Country          object
Product          object
Date             object
Amount           object
Boxes Shipped     int64
dtype: object

Missing Values:
 Sales Person     0
Country          0
Product          0
Date             0
Amount           0
Boxes Shipped    0
dtype: int64


In [3]:
# Fixing it
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df['Amount'] = df['Amount'].replace(r'[\$,]', '', regex=True)
df['Amount'] = pd.to_numeric(df['Amount'])

In [4]:
print("Data Types:\n", df.dtypes)
print("\nMissing Values:\n", df.isna().sum())

Data Types:
 Sales Person             object
Country                  object
Product                  object
Date             datetime64[ns]
Amount                  float64
Boxes Shipped             int64
dtype: object

Missing Values:
 Sales Person     0
Country          0
Product          0
Date             0
Amount           0
Boxes Shipped    0
dtype: int64


In [11]:
# Simulating missing values (as done in the lab) for demonstration
df_missing = df.copy()
df_missing.loc[0:5, 'Amount'] = np.nan

# Strategy: Median Imputation
df_imputed = df_missing.copy()
df_imputed['Amount'] = df_imputed['Amount'].fillna(df_imputed['Amount'].median())

print("Missing values after median imputation:\n", df_imputed.isna().sum())

Missing values after median imputation:
 Sales Person     0
Country          0
Product          0
Date             0
Amount           0
Boxes Shipped    0
dtype: int64


Why Median Imputation?
For this step, **Median Imputation** was selected to handle the missing values in the `Amount` column. Unlike mean imputation, the median is robust to outliers and skewed data. Sales datasets frequently contain extreme values (such as rare, massive bulk orders). Using the median ensures that our imputed values accurately reflect a typical transaction, whereas using the mean could result in artificially inflated values due to those outliers.

In [6]:
# 1. Detect Outliers using IQR
Q1 = df_imputed['Amount'].quantile(0.25)
Q3 = df_imputed['Amount'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Display potential outliers
outliers = df_imputed[(df_imputed['Amount'] < lower_bound) | (df_imputed['Amount'] > upper_bound)]
print(f"Number of outliers detected: {len(outliers)}")

# 2. Handle Outliers (Strategy: Removal)
df_cleaned = df_imputed[(df_imputed['Amount'] >= lower_bound) & (df_imputed['Amount'] <= upper_bound)].copy()

print("Original shape: ", df_imputed.shape)
print("Shape after removing outliers: ", df_cleaned.shape)

Number of outliers detected: 50
Original shape:  (3282, 6)
Shape after removing outliers:  (3232, 6)


In [7]:
# Select numerical columns to scale
features_to_scale = ['Amount', 'Boxes Shipped']

# 1. Min-Max Normalization
min_max_scaler = MinMaxScaler()
df_cleaned[['Amount_MinMax', 'Boxes_MinMax']] = min_max_scaler.fit_transform(df_cleaned[features_to_scale])

# 2. Z-Score Standardization
standard_scaler = StandardScaler()
df_cleaned[['Amount_Zscore', 'Boxes_Zscore']] = standard_scaler.fit_transform(df_cleaned[features_to_scale])

# Display the results
df_cleaned[['Amount', 'Amount_MinMax', 'Amount_Zscore', 'Boxes Shipped', 'Boxes_MinMax', 'Boxes_Zscore']].head()

,Amount,Amount_MinMax,Amount_Zscore,Boxes Shipped,Boxes_MinMax,Boxes_Zscore
0,5220.745,0.29662,-0.14543,180,0.230373,0.121823
1,5220.745,0.29662,-0.14543,94,0.119691,-0.569696
2,5220.745,0.29662,-0.14543,91,0.115830,-0.593819
3,5220.745,0.29662,-0.14543,342,0.438867,1.424452
4,5220.745,0.29662,-0.14543,184,0.235521,0.153987


In [8]:
# Select numerical columns to scale
features_to_scale = ['Amount', 'Boxes Shipped']

# 1. Min-Max Normalization
min_max_scaler = MinMaxScaler()
df_cleaned[['Amount_MinMax', 'Boxes_MinMax']] = min_max_scaler.fit_transform(df_cleaned[features_to_scale])

# 2. Z-Score Standardization
standard_scaler = StandardScaler()
df_cleaned[['Amount_Zscore', 'Boxes_Zscore']] = standard_scaler.fit_transform(df_cleaned[features_to_scale])

# Display the results
df_cleaned[['Amount', 'Amount_MinMax', 'Amount_Zscore', 'Boxes Shipped', 'Boxes_MinMax', 'Boxes_Zscore']].head()

,Amount,Amount_MinMax,Amount_Zscore,Boxes Shipped,Boxes_MinMax,Boxes_Zscore
0,5220.745,0.29662,-0.14543,180,0.230373,0.121823
1,5220.745,0.29662,-0.14543,94,0.119691,-0.569696
2,5220.745,0.29662,-0.14543,91,0.115830,-0.593819
3,5220.745,0.29662,-0.14543,342,0.438867,1.424452
4,5220.745,0.29662,-0.14543,184,0.235521,0.153987


In [9]:
# Use the Z-score standardized features for PCA
X_pca = df_cleaned[['Amount_Zscore', 'Boxes_Zscore']]

# Apply PCA
pca = PCA(n_components=2)
principal_components = pca.fit_transform(X_pca)

# Get explained variance
explained_variance = pca.explained_variance_ratio_
print("Explained Variance Ratio:", explained_variance)

Explained Variance Ratio: [0.50495387 0.49504613]


The PCA resulted in an explained variance ratio of approximately `[0.505, 0.495]`. 

* **Principal Component 1 (PC1)** captures 50.5% of the dataset's variance.
* **Principal Component 2 (PC2)** captures 49.5% of the dataset's variance.

**Interpretation:** Because the variance is split almost exactly 50/50, it tells us that the original features (`Amount` and `Boxes Shipped`) have virtually no linear correlation. In a successful dimensionality reduction, we would expect PC1 to capture a vast majority of the variance (e.g., 85%+), allowing us to drop PC2. However, in this specific case, PCA is not highly effective for reducing dimensions, as dropping either component would mean losing half of the valuable information in the data.